## LSTM-based Recurrent Neural Network

Long Short-Term Memory (LSTM) networks are designed to model sequential data by maintaining a memory of previous tokens.  
Unlike fully connected networks, LSTMs preserve word order and can capture long-range dependencies in text.  
This makes them more suitable for emotion classification tasks where context and negation are important.


In [1]:
import pandas as pd
import numpy as np
import random
import os
import pickle

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report


In [2]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)


In [3]:
BASE_PATH = "../dataset"

train_df = pd.read_csv(os.path.join(BASE_PATH, "train_clean.csv"))
test_df = pd.read_csv(os.path.join(BASE_PATH, "test_clean.csv"))

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()


Train shape: (16000, 2)
Test shape: (2000, 2)


,clean_text,label
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df["label"])
y_test = label_encoder.transform(test_df["label"])

num_classes = len(label_encoder.classes_)
print("Classes:", label_encoder.classes_)


Classes: ['anger' 'fear' 'joy' 'love' 'sadness' 'surprise']


In [5]:
VOCAB_SIZE = 20000
MAX_LEN = 50

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(train_df["clean_text"])

X_train_seq = tokenizer.texts_to_sequences(train_df["clean_text"])
X_test_seq = tokenizer.texts_to_sequences(test_df["clean_text"])

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post")

print("Padded train shape:", X_train_pad.shape)


Padded train shape: (16000, 50)


In [6]:
model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128,
        input_length=MAX_LEN
    ),
    LSTM(128, return_sequences=False),
    Dropout(0.5),
    Dense(num_classes, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 50, 128)           2560000   
                                                                 
 lstm (LSTM)                 (None, 128)               131584    
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense (Dense)               (None, 6)                 774       
                                                                 
Total params: 2692358 (10.27 MB)
Trainable params: 2692358 (10.27 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [7]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/10
225/225 [==============================] - 11s 47ms/step - loss: 1.5943 - accuracy: 0.3237 - val_loss: 1.5708 - val_accuracy: 0.3187
Epoch 2/10
225/225 [==============================] - 10s 46ms/step - loss: 1.5827 - accuracy: 0.3295 - val_loss: 1.5706 - val_accuracy: 0.3187
Epoch 3/10
225/225 [==============================] - 10s 46ms/step - loss: 1.5693 - accuracy: 0.3353 - val_loss: 1.4664 - val_accuracy: 0.3444
Epoch 4/10
225/225 [==============================] - 10s 46ms/step - loss: 1.2287 - accuracy: 0.4105 - val_loss: 1.1502 - val_accuracy: 0.4119
Epoch 5/10
225/225 [==============================] - 10s 46ms/step - loss: 1.0378 - accuracy: 0.4449 - val_loss: 1.0498 - val_accuracy: 0.4450
Epoch 6/10
225/225 [==============================] - 11s 47ms/step - loss: 1.0955 - accuracy: 0.4512 - val_loss: 1.1090 - val_accuracy: 0.4313
Epoch 7/10
225/225 [==============================] - 11s 47ms/step - loss: 0.9569 - accuracy: 0.4844 - val_loss: 0.9840 - val_accuracy:

In [8]:
from sklearn.metrics import accuracy_score, f1_score
y_pred_prob = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Test Accuracy: {acc:.4f}")
print(f"Test F1-score (weighted): {f1:.4f}")


63/63 [==============================] - 1s 8ms/step
Test Accuracy: 0.4965
Test F1-score (weighted): 0.3594


In [9]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

              precision    recall  f1-score   support

       anger       0.51      0.85      0.63       275
        fear       0.00      0.00      0.00       224
         joy       0.51      0.91      0.66       695
        love       0.42      0.79      0.55       159
     sadness       0.50      0.00      0.00       581
    surprise       0.00      0.00      0.00        66

    accuracy                           0.50      2000
   macro avg       0.32      0.42      0.31      2000
weighted avg       0.43      0.50      0.36      2000



### LSTM Model Observations

- The LSTM model outperforms the TF-IDF baseline
- Word order and context improve emotion detection
- Rare emotions still remain challenging due to class imbalance
- Training time is higher than the baseline but manageable


In [10]:
MODEL_PATH = "../models"
os.makedirs(MODEL_PATH, exist_ok=True)

model.save(os.path.join(MODEL_PATH, "lstm_emotion_model.keras"))

with open(os.path.join(MODEL_PATH, "lstm_tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

with open(os.path.join(MODEL_PATH, "lstm_label_encoder.pkl"), "wb") as f:
    pickle.dump(label_encoder, f)

print("LSTM model and artifacts saved.")


LSTM model and artifacts saved.


In [11]:
assert os.path.exists(os.path.join(MODEL_PATH, "lstm_emotion_model.keras"))
assert os.path.exists(os.path.join(MODEL_PATH, "lstm_tokenizer.pkl"))

print("Notebook 3 verified successfully.")


Notebook 3 verified successfully.


### Comparison with Baseline

Compared to the TF-IDF baseline, the LSTM model benefits from
word order and contextual dependencies, leading to improved
performance on several emotion classes.
